# Net worth over time

Reads the latest run file for the current scenario under `data/shared/state/runs/` and plots month-end net worth (nominal) plus an inflation-adjusted (CPI) series.

The scenario defaults the same way the `sim` CLI resolves it: `$ANALYSIS_SCENARIO` if set, else the name recorded in the data repo's `.current_scenario` (written by `sim load <scenario>`), else `current` — use the `Scenario:` dropdown below to pick a different one, then re-run this cell and the ones below.

The run file is opened read-only via `RunConnection`; all reads go through `rc.items(...)`, which returns tidy DataFrames suitable for pandas groupby / pivot. See [ADR-002](../docs/architecture/ADR-002-storage-schema.md) for the underlying schema.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from IPython.display import display, clear_output
from ipywidgets import widgets

from analysis.sim.loader import resolve_scenario_name
from analysis.sim.notebook_ui.pickers import scenario_picker, dollar_plot_mode
from analysis.sim.notebook_ui.networth_plots import (
    net_worth_widget, income_widget, expenses_widget, after_tax_widget,
    gains_widget,
)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", REPO / "data" / "shared" / "sim"))

# Same resolution the CLI uses: $ANALYSIS_SCENARIO, else the data repo's
# .current_scenario, else "current".
_default_scenario = resolve_scenario_name(
    DATA_DIR, os.environ.get("ANALYSIS_SCENARIO") or None
).name

PLOT_MODE = dollar_plot_mode()

# Pick a different scenario here, then re-run this cell and the ones below.
# CONNECTIONS holds the live ScenarioConnection/RunConnection pair; it's
# refreshed in place whenever the dropdown changes.
PICKER, CONNECTIONS = scenario_picker(DATA_DIR, _default_scenario)
display(PICKER)


## Net worth over time


In [ ]:
CONNECTIONS.show(net_worth_widget)

## Income by category

Income flows are transfers _out of_ accounts with `kind = 'income'`. Subkinds are bucketed:

- **salary** — `salary`
- **investment** — `interest`, `dividend`, `capital_gain`
- **other** — `ss_benefit`, `pension`, `rental`, `other_income`


In [ ]:
# Partial-first-year catch-up (salary/investment/SS already received before
# scenario.start) is authored as scenario data (ytd_amount/ytd_preroll on
# scheduled.csv, or year_to_date.csv), so it's already included below --
# the engine posts it as ordinary transfer_event rows (ADR-003 §4.11).
CONNECTIONS.show(income_widget, plot_mode=PLOT_MODE)


## Expenses by category

Expense flows are transfers _into_ accounts with `kind = 'expense'`. Subkinds are bucketed:

- **housing** — `housing`, `interest_expense`
- **healthcare** — `healthcare`
- **taxes** — `taxes_federal`, `taxes_state`, `taxes_property`
- **living** — `living`
- **other** — `other_expense`

The built-in `depreciation` account is excluded: it's a book entry recording value lost by wasting assets (vehicles), not money spent, so counting it here would inflate expenses and understate the after-tax surplus.


In [ ]:
CONNECTIONS.show(expenses_widget, plot_mode=PLOT_MODE)

## Realized vs. unrealized gains

Splits every income-side flow into three buckets, to check whether actual cash distributions (not paper growth) cover expenses:

- **taxable** — reaches a liquid account (`checking`/`savings`) with a taxable character (`ordinary`, `ltcg`, `tax_deferred_out`), plus the *taxable portion* of Social Security.
- **non-taxable** — reaches a liquid account with a `tax_free` character (Roth growth/withdrawals, tax-free income), plus the *non-taxable portion* of Social Security.
- **unrealized** — income-kind growth (interest, dividends, appreciation) that stays inside a retirement/brokerage account and never reaches cash.

Social Security is **not** simply lumped into "taxable": only up to 85% of it ever is (per the federal provisional-income test — see `_taxable_social_security_cents`), and it can be 0% at low enough income. We split each year's SS benefit using the engine's own computed `federal_taxable_ss` (from `tax_year_summary`), not a flat assumption.

Social Security is also counted **gross**: Medicare Part B/D premiums are withheld before the deposit, so those legs go straight from the SS income account to a healthcare expense account and never touch cash. They still count as received (and taxable) income, so `ss_benefit` is bucketed on tax character alone rather than on reaching a liquid account.

Caveats (this is a first pass — flag anything that doesn't match your mental model):

- `return_of_capital` (basis returned on a sale) and `tax_deferred_in` (contributions) are excluded entirely: they're principal movements, not gains.
- "Liquid" = `checking`/`savings` subkinds only; a distribution landing in, say, an HSA cash account wouldn't count as realized here.
- The scenario's final year has no computed `federal_taxable_ss` yet (reconciliation happens the following April), so that year's SS is left entirely in "non-taxable" — an understatement, consistent with the same caveat in "After-tax income" below.
- `distribution_coverage` below is `taxable + non-taxable − expenses`; negative years are ones where realized cash distributions alone didn't cover spending (the gap was plugged by `waterfall_fill`/`refill_transfer` principal draws, which aren't gains and so aren't counted as coverage here). Medicare premiums withheld from Social Security are in both sides of that subtraction, as they should be.


In [ ]:
CONNECTIONS.show(gains_widget, plot_mode=PLOT_MODE)

## After-tax income

After-tax income is gross income less the year's computed federal and state tax liabilities. `tax_free` priors are excluded: they are tax inputs, not a substitute for after-tax income.

> **Tax timing:** tax liability is attributed to the tax year in which income occurs. Withholding and estimated payments are cash movements in that same year; the annual reconciliation payment or refund occurs in April of the following year. Do not interpret the reconciliation year as the tax year.

The engine runs a hidden tax-only tail after the declared scenario end, so the final scenario year has a computed tax liability without displaying a later simulation year.

The final column compares after-tax income with non-tax expenses, so estimated tax liabilities are not counted twice. The chart uses the `PLOT_MODE` selected in the setup cell.

In [ ]:
CONNECTIONS.show(after_tax_widget, plot_mode=PLOT_MODE)